# ML / Statistics Workbench

Run the strict recording-disjoint R²-first workbench and inspect the saved recommendations.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'src').exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from crash_tests.ml_statistics_workbench import MLStatisticsWorkbenchConfig, run_with_config

In [ ]:
RUN_NAME = 'workbench_filtered_full_v1'
SMOKE_RUN = False

config = MLStatisticsWorkbenchConfig(
    parquet_path=repo_root / 'data/raw/33000_ROWS.parquet',
    output_root=repo_root / 'crash_tests/ml_statistics_workbench/outputs',
    run_name=RUN_NAME,
    verbose=True,
)

if SMOKE_RUN:
    config.lightgbm_estimators = 100
    config.xgboost_estimators = 140
    config.mmd_source_train_epochs = 6
    config.mmd_patience = 2
    config.model_selection_splits = 2
    config.latent_candidate_topk = (4, 8)

config

## Run

This cell streams source-encoder logs and writes the new workbench artifacts under the configured output directory.

In [ ]:
output_dir = run_with_config(config)
output_dir

In [ ]:
metrics = pd.read_csv(output_dir / 'metrics.csv')
ablations = pd.read_csv(output_dir / 'feature_ablation_summary.csv')
recommendations = pd.read_csv(output_dir / 'per_target_recommendation.csv')
weights = pd.read_csv(output_dir / 'source_reweighting_summary.csv')
cluster_summary = pd.read_csv(output_dir / 'cluster_summary.csv')
filtered_manifest = pd.read_csv(output_dir / 'filtered_feature_manifest.csv')

display(Markdown('## Recommendations'))
display(recommendations)

display(Markdown('## Best By R²'))
display(metrics.sort_values(['target', 'r2', 'mae'], ascending=[True, False, True]).groupby('target', group_keys=False).head(8))

display(Markdown('## Feature Ablations'))
display(ablations.sort_values(['target', 'cv_r2', 'cv_mae'], ascending=[True, False, True]).groupby('target', group_keys=False).head(12))

display(Markdown('## Source Reweighting'))
display(weights)

display(Markdown('## Cluster Summary'))
display(cluster_summary)

display(Markdown('## Filtered Feature Manifest'))
display(filtered_manifest[filtered_manifest['selected_for_drop']].head(40))

display(Markdown('## Highest Shift Features (Training Side)'))
display(filtered_manifest.sort_values(['feature_set', 'abs_smd'], ascending=[True, False]).groupby('feature_set', group_keys=False).head(20))